In [ ]:
# Function to convert netcdf file into geotiff
import os
import xarray as xr
import rasterio
from rasterio.transform import from_origin
from rasterio.crs import CRS
import numpy as np

def convert_netcdf_to_geotiff_dynamic(folder_path):
    """
    Converts each NetCDF file in the folder to a 4-band GeoTIFF.
    Automatically detects the variable name.
    Each band corresponds to a depth slice, with depth metadata.
    """
    for filename in os.listdir(folder_path):
        if filename.endswith(".nc"):
            file_path = os.path.join(folder_path, filename)
            ds = xr.open_dataset(file_path)

            # Auto-detect the single data variable
            data_vars = list(ds.data_vars)
            if len(data_vars) != 1:
                print(f"Skipping {filename}: expected 1 variable, found {len(data_vars)}")
                continue

            var_name = data_vars[0]
            var_data = ds[var_name]  # shape: (depth, lat, lon)
            data = var_data.values

            if data.ndim != 3 or data.shape[0] != 4:
                print(f"Skipping {filename}: expected shape (4, lat, lon), got {data.shape}")
                continue

            # Get lat/lon
            lat = ds['lat'].values
            lon = ds['lon'].values

            # Flip lat if needed
            if lat[0] < lat[-1]:
                lat = lat[::-1]
                data = data[:, ::-1, :]

            res_lat = abs(lat[1] - lat[0])
            res_lon = abs(lon[1] - lon[0])
            transform = from_origin(lon.min(), lat.max(), res_lon, res_lat)

            # Get depth values and units
            depth_vals = ds['depth'].values
            depth_units = ds['depth'].attrs.get('units', '')

            # Output path
            output_filename = os.path.splitext(filename)[0] + ".tif"
            output_path = os.path.join(folder_path, output_filename)

            # Write GeoTIFF
            with rasterio.open(
                output_path,
                "w",
                driver="GTiff",
                height=data.shape[1],
                width=data.shape[2],
                count=4,
                dtype=data.dtype,
                crs=CRS.from_epsg(4326),
                transform=transform,
            ) as dst:
                for i in range(4):
                    dst.write(data[i, :, :], i + 1)
                    desc = f"{var_name} at Depth: {depth_vals[i]:.2f} {depth_units}".strip()
                    dst.set_band_description(i + 1, desc)

            print(f"✅ Saved: {output_path} with variable '{var_name}' and depth metadata")

In [ ]:
# Path to the input netcdf files
input_nc = r'D:/soil/raw'

# Convert all the netcdf file in the folder into geotiff
convert_netcdf_to_geotiff_dynamic(input_nc)

In [ ]:
# Function to crop and mask geotiff raster
import os
from osgeo import gdal

def crop_and_mask_rasters(input_raster_dir, output_raster_dir, shapefile_path, raster_extensions=(".tif", ".img", ".vrt")):
    """
    Crops and masks all raster files in a directory using a shapefile.
    Saves output rasters with a 'CanTransBasin_' prefix in the output directory.

    Parameters:
    - input_raster_dir (str): Path to the folder containing input rasters.
    - output_raster_dir (str): Path to the folder where output rasters will be saved.
    - shapefile_path (str): Path to the shapefile used for cropping/masking.
    - raster_extensions (tuple): File extensions to include (default: .tif, .img, .vrt).
    """
    os.makedirs(output_raster_dir, exist_ok=True)

    for raster_file in os.listdir(input_raster_dir):
        if raster_file.endswith(raster_extensions):
            input_raster_path = os.path.join(input_raster_dir, raster_file)
            output_raster_path = os.path.join(output_raster_dir, f"CanTransBasin_{raster_file}")

            # Open raster and read NoData value from first band
            dataset = gdal.Open(input_raster_path)
            band = dataset.GetRasterBand(1)
            nodata_value = band.GetNoDataValue()

            warp_options = {
                'format': 'GTiff',
                'cutlineDSName': shapefile_path,
                'cropToCutline': True,
                'creationOptions': ["COMPRESS=DEFLATE"]
            }

            # Only include dstNodata if the input raster has it defined
            if nodata_value is not None:
                warp_options['dstNodata'] = nodata_value

            gdal.Warp(output_raster_path, input_raster_path, **warp_options)

            print(f"✅ Processed: {raster_file} → {output_raster_path}")

    print("🎉 All rasters processed successfully!")

In [ ]:
# Path to root directory 
root_dir = r'D:'
# Input directory containing raster files
input_raster_dir_path = os.path.join(root_dir, "soil", "raw")
# Output directory for cropped and masked rasters
output_raster_dir_path = os.path.join(root_dir, "soil", "CanTrans")
# Path to the shapefile used for cropping and masking
input_shapefile_path = os.path.join(root_dir, "study_domain", "CanTrans_BasinBoundary_MERIT_withBuffer.shp") 

# Crop and mask the inputs by the give shapefile domain
crop_and_mask_rasters(
    input_raster_dir=input_raster_dir_path,
    output_raster_dir=output_raster_dir_path,
    shapefile_path=input_shapefile_path
)

In [ ]:
# Function that combine many geotiff file into a single multiband geotiff
import glob
import os
import numpy as np
import rasterio

def combine_geotiffs(pattern, output_path):
    file_list = glob.glob(pattern)

    if not file_list:
        print("❌ No matching GeoTIFF files found.")
        return

    # Delete existing output file if it exists (to avoid read errors)
    if os.path.exists(output_path):
        os.remove(output_path)

    src_files = [rasterio.open(fp) for fp in file_list]

    # Read all bands from all files
    all_bands = []
    for src in src_files:
        for i in range(1, src.count + 1):
            band = src.read(i)
            all_bands.append(band)

    # Stack all bands into a single array
    stacked_array = np.stack(all_bands)

    # Use metadata from the first file
    ref = src_files[0]
    meta = ref.meta.copy()
    meta.update({
        "count": stacked_array.shape[0],
        "driver": "GTiff",
        "compress": "DEFLATE"  # Apply LZW compression
    })

    # Write to output file
    with rasterio.open(output_path, "w", **meta) as dst:
        for i in range(stacked_array.shape[0]):
            dst.write(stacked_array[i], i + 1)

    print(f"✅ Combined raster saved to: {output_path}")

In [ ]:
# Excute function to combine individual geotiff into a single multiband geotiff
combine_geotiffs(
    pattern=r'D:/soil/CanTrans/*CLAY*.tif',
    output_path=r'D:/soil/CanTransBasin_CLAY.tif'
)
combine_geotiffs(
    pattern=r'D:/soil/CanTrans/*SAND*.tif',
    output_path=r'D:/soil/CanTransBasin_SAND.tif'
)
combine_geotiffs(
    pattern=r'D:/soil/CanTrans/*OC*.tif',
    output_path=r'D:/soil/CanTransBasin_OC.tif'
)

In [ ]:
# Function to calculate Soil parameters for MESH interval soil layer:
import rasterio
import numpy as np
import os
import glob
import csv
from tqdm import tqdm
from datetime import datetime

# Calculate normalized weights for each mesh interval
def calculate_weights(gsde_intervals, mesh_intervals):
    weights_used = []
    for i, (mesh_start, mesh_end) in enumerate(mesh_intervals):
        weights = []
        for gsde_start, gsde_end in gsde_intervals:
            overlap_start = max(mesh_start, gsde_start)
            overlap_end = min(mesh_end, gsde_end)
            weight = (overlap_end - overlap_start) / (mesh_end - mesh_start) if overlap_start < overlap_end else 0
            weights.append(weight)
        total = sum(weights)
        if total == 0:
            print(f"⚠️ Mesh interval {mesh_intervals[i]} has zero overlap with GSDE layers.")
        normalized = [w / total for w in weights] if total > 0 else [0] * len(weights)
        weights_used.append(normalized)
    return np.array(weights_used)  # Shape: (M, 8)

# Save weights to timestamped CSV
def save_weights_to_csv(weights_used, gsde_intervals, mesh_intervals, output_folder, project_name="soil_weights"):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    csv_name = f"{project_name}_weights_{timestamp}.csv"
    csv_path = os.path.join(output_folder, csv_name)

    with open(csv_path, mode="w", newline="") as file:
        writer = csv.writer(file)
        header = ["Mesh Interval"] + [f"{start}–{end} m" for start, end in gsde_intervals]
        writer.writerow(header)

        for i, weight_row in enumerate(weights_used):
            mesh_label = f"{mesh_intervals[i][0]}–{mesh_intervals[i][1]} m"
            writer.writerow([mesh_label] + [round(w, 6) for w in weight_row])

    print(f"📄 Weights saved to: {csv_path}")

# Apply weights with vectorized NoData-aware logic
def apply_weights_to_geotiff(input_path, output_path, mesh_intervals, weights_used):
    if os.path.exists(output_path):
        os.remove(output_path)

    with rasterio.open(input_path) as src:
        nodata = src.nodata
        bands = np.stack([
            src.read(i).astype(np.float32)
            for i in range(1, src.count + 1)
        ])  # Shape: (8, H, W)

        if nodata is not None:
            bands = np.where(bands == nodata, np.nan, bands)

        B, H, W = bands.shape
        bands_2d = bands.reshape(B, -1)  # Shape: (8, H*W)
        valid_mask = ~np.isnan(bands_2d)
        safe_bands = np.nan_to_num(bands_2d, nan=0.0)

        weighted_bands = []

        for i, weight_set in enumerate(weights_used):
            weights = np.array(weight_set).reshape(B, 1)
            masked_weights = weights * valid_mask
            weight_sums = masked_weights.sum(axis=0)
            normalized_weights = np.divide(
                masked_weights, weight_sums, where=weight_sums != 0
            )
            weighted_avg = np.einsum('ij,ij->j', normalized_weights, safe_bands)
            weighted_avg = weighted_avg.reshape(H, W).astype(np.float32)
            weighted_avg[weight_sums.reshape(H, W) == 0] = np.nan
            weighted_bands.append(weighted_avg)

        output_array = np.stack(weighted_bands)

        meta = src.meta.copy()
        meta.update({
            "count": len(mesh_intervals),
            "dtype": "float32",
            "driver": "GTiff",
            "compress": "DEFLATE",
            "nodata": np.nan
        })

        with rasterio.open(output_path, "w", **meta) as dst:
            for i, band_data in enumerate(output_array):
                dst.write(band_data, i + 1)
                depth_range = f"{mesh_intervals[i][0]}–{mesh_intervals[i][1]} m"
                dst.set_band_description(i + 1, f"Weighted avg for {depth_range}")

    print(f"✅ Saved: {output_path}")

# Batch process all soil properties
def process_soil_folder(folder_path, gsde_intervals, mesh_intervals, soil_properties):
    weights_used = calculate_weights(gsde_intervals, mesh_intervals)
    save_weights_to_csv(weights_used, gsde_intervals, mesh_intervals, folder_path)

    for prop in tqdm(soil_properties, desc="Processing soil properties"):
        pattern = os.path.join(folder_path, f"*{prop}*.tif")
        matches = glob.glob(pattern)
        if not matches:
            print(f"❌ No file found for {prop}")
            continue

        input_path = matches[0]
        output_path = os.path.join(folder_path, f"{os.path.splitext(os.path.basename(input_path))[0]}_mesh_weighted.tif")
        apply_weights_to_geotiff(input_path, output_path, mesh_intervals, weights_used)

In [ ]:
# Depth intervals (GSDE)
gsde_intervals = [(0, 0.045), (0.045, 0.091), (0.091, 0.166), (0.166, 0.289),
                  (0.289, 0.493), (0.493, 0.829), (0.829, 1.383), (1.383, 2.296)]
# Mesh intervals (can be any length)
mesh_intervals = [(0, 0.1), (0.1, 0.35), (0.35, 1.2), (1.2, 4.1)]
# Soil properties to process
soil_properties = ["CLAY", "SAND", "OC"]
input_tif = r"D:/soil"  
process_soil_folder(input_tif, gsde_intervals, mesh_intervals, soil_properties)

In [ ]:
# Function to rename the raster
import rasterio
from rasterio.enums import Resampling
from osgeo import gdal
import os

def rename_bands(input_raster_path, output_raster_path, new_band_names):
    # Open the original raster using GDAL
    ds = gdal.Open(input_raster_path, gdal.GA_ReadOnly)
    if ds is None:
        raise FileNotFoundError(f"Unable to open {input_raster_path}")
    
    band_count = ds.RasterCount

    if len(new_band_names) != band_count:
        raise ValueError(f"Provided {len(new_band_names)} band names, but raster has {band_count} bands.")

    # Create a copy of the original raster
    driver = gdal.GetDriverByName('GTiff')
    output_ds = driver.CreateCopy(output_raster_path, ds, strict=0)
    
    # Rename each band
    for i in range(band_count):
        band = output_ds.GetRasterBand(i + 1)
        band.SetDescription(new_band_names[i])

    # Flush and close
    output_ds.FlushCache()
    output_ds = None
    ds = None
    print(f"Saved renamed raster to {output_raster_path}")

In [ ]:
# Example usage
if __name__ == "__main__":
    input_raster = r'D:/soil/CanTransBasin_SAND_mesh_weighted.tif'          # Replace with your input file path
    output_raster = r'D:/soil/CanTransBasin_SAND_mesh_weighted_layer.tif'       # Replace with your desired output file path
    band_names = ["Layer1", "Layer2", "Layer3", "Layer4"]  # Customize for your specific bands
    rename_bands(input_raster, output_raster, band_names)

In [63]:
# Function to convert netcdf file into geotiff
import os
import xarray as xr
import rasterio
from rasterio.transform import from_origin
from rasterio.crs import CRS
import numpy as np
from exactextract import exact_extract
output_shapefile = r'D:/MERIT/agg_MERIT_CanTrans_subbasins.shp'
output_shapefile2 = r'D:/MERIT/agg_geomfixed_MERIT_CanTrans_subbasins.shp'
sand_path = r'D:/soil/CanTransBasin_SAND_mesh_weighted.tif'
clay_path = r'D:/soil/CanTransBasin_CLAY_mesh_weighted.tif'
oc_path = r'D:/soil/CanTransBasin_OC_mesh_weighted.tif'
output_csv = r'D:/soil/CanTrans_model_stats_soil.csv'

## zonal statistic  
results1 = exact_extract(sand_path, output_shapefile2, 'mean', include_cols=['COMID'], output='pandas')
results2 = exact_extract(clay_path, output_shapefile2, 'mean', include_cols=['COMID'], output='pandas')
results3 = exact_extract(oc_path, output_shapefile2, 'mean', include_cols=['COMID'], output='pandas')

## Rename the columns
def rename_bands(df, prefix):
    df.rename(columns={f"band_{i}_mean": f"{prefix}{i}" for i in range(1, 5)}, inplace=True)

## Apply renaming
rename_bands(results1, "meshSAND")
rename_bands(results2, "meshCLAY")
rename_bands(results3, "meshOC")

# Join the dataframe
results1 = results1.merge(results2[['COMID', 'meshCLAY1', 'meshCLAY2', 'meshCLAY3', 'meshCLAY4']], on='COMID', how='left')
results1 = results1.merge(results3[['COMID', 'meshOC1', 'meshOC2', 'meshOC3', 'meshOC4']], on='COMID', how='left')

# Save the selected columns to a CSV file
results1.to_csv(output_csv, index=False)
#
gdf = gpd.read_file(output_shapefile2)
gdf = gdf.merge(results1, on='COMID', how='left')
gdf.to_file(input_basin, driver='ESRI Shapefile')

results1.to_csv(output_csv, index=False)

In [ ]:
# Fix shapefile
import geopandas as gpd
from shapely.validation import explain_validity
def fix_and_save_valid_shapefile(input_path, output_path):
    try:
        gdf = gpd.read_file(input_path)
    except Exception as e:
        print(f"Error reading file: {e}")
        return

    # Check how many are invalid
    invalid_count = (~gdf.is_valid).sum()
    print(f"Found {invalid_count} invalid geometries.")

    # Print reasons for invalidity (optional)
    for idx, row in gdf[~gdf.is_valid].iterrows():
        print(f"Feature ID {idx}: {explain_validity(row.geometry)}")

    # Fix invalid geometries
    gdf['geometry'] = gdf['geometry'].apply(
        lambda geom: geom if geom.is_valid else geom.buffer(0)
    )

    # Confirm all geometries are now valid
    if gdf.is_valid.all():
        print("All geometries fixed and now valid.")
    else:
        print("Some geometries are still invalid.")

    # Save to new shapefile
    gdf.to_file(output_path)
    print(f"Fixed shapefile saved to: {output_path}")

# Example usage
fix_and_save_valid_shapefile(output_shapefile, output_shapefile2)